In [0]:
from pyspark.sql.functions import col, regexp_extract, when, lit, trim, lower, regexp_replace
from pyspark.sql.types import IntegerType
from delta.tables import DeltaTable
import datetime, json

# ---------- Setup ----------
catalog = "db_dataclassdev"
schema = "silver"
table = "item"
qualified_name = f"{catalog}.{schema}.{table}"
silver_path = f"abfss://silver@strdatabrickssadls.dfs.core.windows.net/{table}"
current_time = datetime.datetime.utcnow()

# ---------- Load & Transform ----------
df = spark.read.format("delta").table("db_dataclassdev.bronze.item")

if "descrption" in df.columns:
    df = df.withColumnRenamed("descrption", "description")

df = df.withColumn("code", col("code").cast("long")) \
       .withColumn("type", regexp_extract("type", r"\d+", 0).cast(IntegerType())) \
       .withColumn("clean_size", trim(lower(regexp_replace(col("size"), ",", "")))) \
       .withColumn("lb_val", regexp_extract(col("clean_size"), r"(\d+(\.\d+)?)\s*lb", 1).cast("double")) \
       .withColumn("oz_val", regexp_extract(col("clean_size"), r"(\d+(\.\d+)?)\s*(oz|ounce)", 1).cast("double")) \
       .withColumn("size_in_oz",
            when(col("lb_val").isNotNull() & col("oz_val").isNotNull(), col("lb_val") * 16 + col("oz_val"))
           .when(col("lb_val").isNotNull(), col("lb_val") * 16)
           .when(col("oz_val").isNotNull(), col("oz_val"))
           .otherwise(None)
       ) \
       .withColumn("specialsize",
            when(col("lb_val").isNull() & col("oz_val").isNull(), col("clean_size"))
           .otherwise(lit(None))
       ) \
       .withColumn("etl_record_created_date", lit(current_time)) \
       .withColumn("etl_record_modified_date", lit(current_time)) \
       .withColumn("isActive", lit(True)) \
       .withColumn("start_date", lit(current_time)) \
       .withColumn("end_date", lit(None).cast("timestamp"))

# ---------- Final Selected Columns ----------
df_silver = df.select(
    "code", "description", "type", "brand", "size", "size_in_oz", "specialsize",
    "etl_record_created_date", "etl_record_modified_date",
    "isActive", "start_date", "end_date"
).filter("code IS NOT NULL")  # remove rows with null code

# ---------- SCD Type 2 MERGE ----------
if DeltaTable.isDeltaTable(spark, silver_path):
    target_tbl = DeltaTable.forPath(spark, silver_path)
    
    updates = df_silver.alias("updates")
    target = target_tbl.alias("target")

    target_tbl.alias("target").merge(
        updates,
        "target.code = updates.code AND target.isActive = true"
    ).whenMatchedUpdate(
        condition="""target.size_in_oz != updates.size_in_oz OR 
                     target.description != updates.description OR 
                     target.type != updates.type OR 
                     target.brand != updates.brand""",
        set={
            "isActive": lit(False),
            "end_date": lit(current_time)
        }
    ).whenNotMatchedInsert(
        values={
            "code": col("updates.code"),
            "description": col("updates.description"),
            "type": col("updates.type"),
            "brand": col("updates.brand"),
            "size": col("updates.size"),
            "size_in_oz": col("updates.size_in_oz"),
            "specialsize": col("updates.specialsize"),
            "etl_record_created_date": col("updates.etl_record_created_date"),
            "etl_record_modified_date": col("updates.etl_record_modified_date"),
            "isActive": col("updates.isActive"),
            "start_date": col("updates.start_date"),
            "end_date": col("updates.end_date")
        }
    ).execute()

else:
    df_silver.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("path", silver_path) \
        .saveAsTable(qualified_name)

# ---------- Audit Info ----------
active_count = spark.read.format("delta").load(silver_path).filter("isActive = true").count()

audit_info = {
    "fileName": "item",
    "rowCount": active_count,
    "status": "Succeeded",
    "destinationPath": silver_path,
    "ucTable": qualified_name,
    "timestamp": str(current_time)
}

dbutils.notebook.exit(json.dumps(audit_info))
